In [234]:
categories = [
    "Electricity", "Water", "Sanitation", "Roads", "Telecom",
    "Banking", "Insurance", "Pension", "Education", "Healthcare",
    "Transport", "Documents", "Housing", "Employment", "Law & Order",
    "Railways", "Gas", "Internet Services", "Tax", "Other"
]

In [235]:
import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")

# Keep useful columns
df = df[["Ticket Subject", "Ticket Description", "Ticket Type"]]

# Combine text
df["text"] = df["Ticket Subject"].fillna('') + " " + df["Ticket Description"].fillna('')

# Rename
df = df[["text", "Ticket Type"]]
df = df.rename(columns={"Ticket Type": "category"})

# Drop empty
df = df.dropna()
df = df[df["text"].str.strip() != ""]

In [236]:
def map_kaggle_category(label):
    label = str(label).lower()
    
    if "technical" in label:
        return "Internet Services"
    elif "billing" in label:
        return "Banking"
    elif "account" in label:
        return "Documents"
    else:
        return "Other"

df["category"] = df["category"].apply(map_kaggle_category)

In [237]:
import random

synthetic_data = []

templates = {
    "Electricity": [
        "No electricity for {} days",
        "Power cut in my area",
        "Transformer not working",
        "Voltage fluctuation issue"
    ],
    "Water": [
        "No water supply for {} days",
        "Water leakage in street",
        "Dirty water coming from taps"
    ],
    "Sanitation": [
        "Garbage not collected",
        "Street is very dirty",
        "Waste not cleared from area"
    ],
    "Roads": [
        "Road has potholes",
        "Street is damaged",
        "Bridge condition is bad"
    ],
    "Telecom": [
        "Mobile network not working",
        "Call drops frequently",
        "No signal in my area"
    ],
    "Banking": [
        "Money deducted from account",
        "Bank transaction failed",
        "ATM not working"
    ],
    "Insurance": [
        "Insurance claim not processed",
        "Policy issue",
        "Claim rejected"
    ],
    "Pension": [
        "Pension not received",
        "Delay in pension payment"
    ],
    "Education": [
        "School facilities are poor",
        "College administration issue"
    ],
    "Healthcare": [
        "Hospital staff not available",
        "Medical service issue"
    ],
    "Transport": [
        "Bus service not available",
        "Public transport delay"
    ],
    "Documents": [
        "Passport not received",
        "Aadhaar issue",
        "Certificate delay"
    ],
    "Housing": [
        "Housing scheme issue",
        "Flat allocation problem"
    ],
    "Employment": [
        "Job application issue",
        "Salary not received"
    ],
    "Law & Order": [
        "Police not responding",
        "Crime in my area"
    ],
    "Railways": [
        "Train delayed",
        "Railway complaint"
    ],
    "Gas": [
        "Gas cylinder not delivered",
        "Gas leakage issue"
    ],
    "Internet Services": [
        "Internet not working",
        "WiFi issue"
    ],
    "Tax": [
        "Tax refund not received",
        "Tax calculation issue"
    ]
}

In [238]:
for category, temp_list in templates.items():
    for _ in range(50):
        sentence = random.choice(temp_list).format(random.randint(1,5))
        synthetic_data.append([sentence, category])

In [239]:
synthetic_df = pd.DataFrame(synthetic_data, columns=["text", "category"])

In [240]:
df = pd.concat([df, synthetic_df], ignore_index=True)

In [241]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df["clean_text"] = df["text"].apply(preprocess)

In [242]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])
y = df["category"]

In [243]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

LogisticRegression(max_iter=200)

In [244]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

          Banking       0.70      0.04      0.08       334
        Documents       1.00      1.00      1.00        11
        Education       1.00      1.00      1.00        13
      Electricity       1.00      1.00      1.00        10
       Employment       1.00      1.00      1.00         5
              Gas       1.00      1.00      1.00         7
       Healthcare       1.00      1.00      1.00        11
          Housing       1.00      1.00      1.00        11
        Insurance       1.00      1.00      1.00        10
Internet Services       0.62      0.05      0.09       373
      Law & Order       1.00      1.00      1.00        14
            Other       0.60      0.99      0.75      1017
          Pension       1.00      1.00      1.00         7
         Railways       1.00      1.00      1.00         7
            Roads       1.00      1.00      1.00         8
       Sanitation       1.00      1.00      1.00       

In [246]:
keyword_map = {

    "Electricity": [
        "electricity", "power", "transformer", "voltage", "current",
        "power cut", "outage", "line", "meter","light"
    ],

    "Water": [
        "water", "supply", "leakage", "pipe", "tap",
        "drainage", "sewage", "dirty water"
    ],

    "Sanitation": [
        "garbage", "waste", "dirty", "clean", "sanitation",
        "dustbin", "sewer", "trash"
    ],

    "Roads": [
        "road", "pothole", "bridge", "street", "highway",
        "construction", "damage"
    ],

    "Telecom": [
        "mobile", "network", "signal", "tower", "call",
        "sim", "coverage", "call drop", "net"
    ],

    "Banking": [
        "bank", "account", "atm", "transaction", "money",
        "balance", "debit", "credit", "payment"
    ],

    "Insurance": [
        "insurance", "policy", "claim", "premium",
        "coverage", "insured"
    ],

    "Pension": [
        "pension", "retirement", "fund", "pf",
        "provident fund"
    ],

    "Education": [
        "school", "college", "education", "teacher",
        "student", "exam", "university", "fees"
    ],

    "Healthcare": [
        "hospital", "doctor", "medical", "health",
        "treatment", "medicine", "clinic"
    ],

    "Transport": [
        "bus", "transport", "vehicle", "traffic",
        "public transport", "auto", "taxi"
    ],

    "Documents": [
        "passport", "aadhaar", "certificate", "document",
        "license", "id card", "verification"
    ],

    "Housing": [
        "house", "housing", "flat", "rent",
        "apartment", "property"
    ],

    "Employment": [
        "job", "employment", "salary", "wage",
        "recruitment", "vacancy"
    ],

    "Law & Order": [
        "police", "crime", "theft", "fraud",
        "law", "complaint", "harassment"
    ],

    "Railways": [
        "train", "railway", "ticket", "station",
        "platform", "irctc"
    ],

    "Gas": [
        "gas", "cylinder", "lpg", "leakage",
        "refill"
    ],

    "Internet Services": [
        "internet", "wifi", "broadband", "router",
        "connection", "speed", "lag"
    ],

    "Tax": [
        "tax", "gst", "income tax", "refund",
        "return", "filing"
    ],

    "Other": []  # fallback category
}

In [247]:
def keyword_predict(text):
    text = text.lower()
    
    scores = {}

    for category, keywords in keyword_map.items():
        scores[category] = 0
        for word in keywords:
            if word in text:
                scores[category] += 1

    # Get best match
    best_category = max(scores, key=scores.get)

    if scores[best_category] > 0:
        return best_category
    
    return None

In [248]:
def multi_keyword_predict(text):
    text = text.lower()
    
    matched_categories = []

    for category, keywords in keyword_map.items():
        for word in keywords:
            if word in text:
                matched_categories.append(category)
                break

    return list(set(matched_categories))  # remove duplicates

In [249]:
def predict_multi(text):
    
    keyword_results = multi_keyword_predict(text)
    
    if len(keyword_results) > 0:
        return keyword_results
    
    # fallback to ML
    text_clean = preprocess(text)
    vec = vectorizer.transform([text_clean])
    
    pred = model.predict(vec)[0]
    
    return [pred]

In [250]:
print(predict_multi("Internet is not working due to water leakage and short circuit"))

['Telecom', 'Gas', 'Internet Services', 'Water']


In [251]:
df_balanced = df.groupby('category').apply(lambda x: x.sample(n=150, replace=True)).reset_index(drop=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_19668\3598293483.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('category').apply(lambda x: x.sample(n=150, replace=True)).reset_index(drop=True)


In [252]:
other_df = df[df["category"] == "Other"].sample(150)
rest_df = df[df["category"] != "Other"]

df = pd.concat([rest_df, other_df])

In [253]:
model = LogisticRegression(max_iter=200, class_weight='balanced')

In [254]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)  # BIG IMPROVEMENT
)

In [255]:
synthetic_data = [

# Electricity
["Electricity has been gone since last night", "Electricity"],
["Facing frequent power cuts in my locality", "Electricity"],
["Voltage is fluctuating continuously in our area", "Electricity"],
["Transformer near my house is sparking", "Electricity"],
["No light in the street for past 2 days", "Electricity"],
["Power outage happens every evening", "Electricity"],
["Meter is not working properly", "Electricity"],
["Electric line has fallen on the road", "Electricity"],

# Water
["Water supply is not coming from yesterday", "Water"],
["There is continuous water leakage near my home", "Water"],
["Dirty water is coming from taps", "Water"],
["Water pressure is very low in our area", "Water"],
["Pipeline is broken and water is flooding the street", "Water"],
["No drinking water available in our locality", "Water"],
["Water tank is not being filled properly", "Water"],

# Sanitation
["Garbage has not been collected for many days", "Sanitation"],
["Area is very dirty and smells bad", "Sanitation"],
["Waste is lying on the road for a week", "Sanitation"],
["No dustbins available in our street", "Sanitation"],
["Drainage is blocked and causing issues", "Sanitation"],
["Street cleaning is not happening regularly", "Sanitation"],

# Roads
["Road condition is very खराब with many potholes", "Roads"],
["Street is broken and not repaired since months", "Roads"],
["Bridge is damaged and unsafe for travel", "Roads"],
["Road construction work is incomplete", "Roads"],
["Heavy potholes causing traffic issues", "Roads"],
["Gali ki road bilkul kharab hai", "Roads"],

# Telecom
["Mobile network is not available in my area", "Telecom"],
["Call drops happening frequently", "Telecom"],
["Signal strength is very weak", "Telecom"],
["Unable to make calls properly", "Telecom"],
["Network tower seems down", "Telecom"],

# Banking
["Money deducted but not credited to account", "Banking"],
["Transaction failed but amount debited", "Banking"],
["ATM is not dispensing cash", "Banking"],
["Unable to check account balance", "Banking"],
["Online payment failed multiple times", "Banking"],
["Bank server is always down", "Banking"],

# Insurance
["Insurance claim is still pending from months", "Insurance"],
["Policy details are incorrect", "Insurance"],
["Claim has been rejected without reason", "Insurance"],
["Premium payment not updated", "Insurance"],

# Pension
["Pension has not been credited this month", "Pension"],
["Delay in receiving pension amount", "Pension"],
["Retirement benefits not processed yet", "Pension"],

# Education
["School teachers are not attending classes", "Education"],
["College administration is not responding", "Education"],
["Exam results are delayed", "Education"],
["Fees issue not resolved", "Education"],

# Healthcare
["Hospital staff is not available", "Healthcare"],
["Doctor is not present during duty hours", "Healthcare"],
["No proper medical facilities in clinic", "Healthcare"],
["Medicines are not available in hospital", "Healthcare"],

# Transport
["Bus service is irregular and late", "Transport"],
["Public transport is not available on time", "Transport"],
["Traffic congestion is too high", "Transport"],
["Auto drivers are refusing rides", "Transport"],

# Documents
["Passport application is still pending", "Documents"],
["Aadhaar card update not done", "Documents"],
["Certificate verification is delayed", "Documents"],
["Driving license not received yet", "Documents"],

# Housing
["Housing scheme allotment delayed", "Housing"],
["Flat possession is not given", "Housing"],
["Building construction is incomplete", "Housing"],

# Employment
["Salary has not been credited yet", "Employment"],
["Job application status not updated", "Employment"],
["Recruitment process is delayed", "Employment"],

# Law & Order
["Police is not taking any action", "Law & Order"],
["Theft happened in my area", "Law & Order"],
["Fraud case not registered", "Law & Order"],
["Harassment complaint ignored", "Law & Order"],

# Railways
["Train is running late by 4 hours", "Railways"],
["Ticket booking failed", "Railways"],
["Railway staff not cooperating", "Railways"],

# Gas
["Gas cylinder not delivered on time", "Gas"],
["Gas leakage smell coming from cylinder", "Gas"],
["Refill request is still pending", "Gas"],

# Internet Services
["Internet is very slow and disconnecting", "Internet Services"],
["WiFi is not working properly", "Internet Services"],
["Broadband connection is down", "Internet Services"],
["Router keeps restarting", "Internet Services"],

# Tax
["Tax refund not received yet", "Tax"],
["GST filing issue", "Tax"],
["Incorrect tax calculation", "Tax"],

# Other (important!)
["Something is wrong but I cannot explain", "Other"],
["System is behaving strangely", "Other"],
["I am facing unknown issue", "Other"],
["Problem is unclear please check", "Other"]
]

In [256]:
synthetic_df = pd.DataFrame(synthetic_data, columns=["text", "category"])

df = pd.concat([df, synthetic_df], ignore_index=True)

In [257]:
import random

augmented_data = []

for text, category in synthetic_data:
    for _ in range(5):  # increase data 5x
        new_text = text + " please resolve urgently"
        augmented_data.append([new_text, category])

aug_df = pd.DataFrame(augmented_data, columns=["text", "category"])

df = pd.concat([df, aug_df], ignore_index=True)

In [258]:
other_df = df[df["category"] == "Other"].sample(100, random_state=42)
rest_df = df[df["category"] != "Other"]

df = pd.concat([rest_df, other_df])

In [259]:
df_balanced = df.groupby('category').apply(
    lambda x: x.sample(n=150, replace=True, random_state=42)
).reset_index(drop=True)

df = df_balanced

C:\Users\HP\AppData\Local\Temp\ipykernel_19668\3183938575.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('category').apply(


In [260]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   # VERY IMPORTANT
    min_df=2             # removes noise
)

In [261]:
model = LogisticRegression(
    max_iter=300,
    class_weight='balanced'
)

In [262]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6342887473460722


In [265]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

          Banking       0.70      0.04      0.08       334
        Documents       1.00      1.00      1.00        11
        Education       1.00      1.00      1.00        13
      Electricity       1.00      1.00      1.00        10
       Employment       1.00      1.00      1.00         5
              Gas       1.00      1.00      1.00         7
       Healthcare       1.00      1.00      1.00        11
          Housing       1.00      1.00      1.00        11
        Insurance       1.00      1.00      1.00        10
Internet Services       0.62      0.05      0.09       373
      Law & Order       1.00      1.00      1.00        14
            Other       0.60      0.99      0.75      1017
          Pension       1.00      1.00      1.00         7
         Railways       1.00      1.00      1.00         7
            Roads       1.00      1.00      1.00         8
       Sanitation       1.00      1.00      1.00       

In [264]:
print(df["category"].value_counts())

category
Banking              150
Documents            150
Transport            150
Telecom              150
Tax                  150
Sanitation           150
Roads                150
Railways             150
Pension              150
Other                150
Law & Order          150
Internet Services    150
Insurance            150
Housing              150
Healthcare           150
Gas                  150
Employment           150
Electricity          150
Education            150
Water                150
Name: count, dtype: int64


def interactive_test():
    print("\nType 'exit' to stop\n")
    
    while True:
        user_input = input("Enter complaint: ")
        
        if user_input.lower() == "exit":
            break
        
        result = predict(user_input)
        print(f"Predicted Category: {result}")
        print("-" * 40)

# Run this
interactive_test()